In [1]:
from ingest.neo4j import create_nodes
from config import neo4j
import json

graph = neo4j.load_neo4j_graph()

In [2]:
file_names = ["Talleyrand", "Napoleon", "Battle_of_Waterloo"]

In [3]:
for name in file_names:
    file = f"datadocs/{name}.json"

    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if name == "Battle_of_Waterloo":
        create_nodes(graph=graph, data=data, node_label="Event", node_name=name)
    else:
        create_nodes(graph=graph, data=data, node_label="Person", node_name=name)

In [4]:
rel_person_person = """
MATCH (p1:Person), (p2:Person)
WHERE elementId(p1) < elementId(p2)
MERGE (p1)-[:RELATED_TO]->(p2)
MERGE (p2)-[:RELATED_TO]->(p1);
"""
rel_person_event = """
MATCH (p:Person), (e:Event)
MERGE (p)-[:RELATED_TO]->(e)
MERGE (e)-[:RELATED_TO]->(p);
"""

rel_person_section = """
MATCH (p:Person), (s:Section)
WHERE p.name = s.parent_name
MERGE (p)-[:HAS_SECTION]->(s);
"""
rel_event_section = """
MATCH (e:Event), (s:Section)
WHERE e.name = s.parent_name
MERGE (e)-[:HAS_SECTION]->(s);
"""

queries = [rel_person_person, rel_person_event, rel_person_section, rel_event_section]

for query in queries:
    graph.query(query)

print("All relationships created successfully.")

All relationships created successfully.


#### Step 5 — Ingest into Pinecone: Vector Embeddings

In [5]:
# from ingest.pinecone_ingest import process_and_upsert
# process_and_upsert(file_names)

#### Step 6 — Define the Graph Traversal Query

In [6]:
query_search = """
MATCH (n)-[r]-(m)
WHERE n.name = $name
RETURN 
    n AS matchedNode,
    r AS relationship,
    m AS relatedNode
"""


In [8]:
from retrieve.neo4j_pinecone import search_and_fetch
search_and_fetch(query_search, query_text="who killed nepolian?")

[{'score': 0.262633026,
  'metadata': {'chunk_index': 147.0,
   'name': 'Battle_of_Waterloo_info',
   'section': 'Consequence',
   'text': 'by the British archaeologist Tony Pollard, concluded that in the aftermath of the conflict, local farmers dug up the corpses of horses and men and sold them to the Waterloo sugar factory. There, the ground-down bones were fired in kilns to make bone-char, which was then used to filter sugar syrup as part of the production process.'},
  'neo4j_nodes': [{'matchedNode': {'name': 'Battle_of_Waterloo_info'},
    'relationship': ({'name': 'Battle_of_Waterloo_info'},
     'RELATED_TO',
     {'name': 'Talleyrand_info'}),
    'relatedNode': {'name': 'Talleyrand_info'}},
   {'matchedNode': {'name': 'Battle_of_Waterloo_info'},
    'relationship': ({'name': 'Battle_of_Waterloo_info'},
     'RELATED_TO',
     {'name': 'Napoleon_info'}),
    'relatedNode': {'name': 'Napoleon_info'}},
   {'matchedNode': {'name': 'Battle_of_Waterloo_info'},
    'relationship': ({'